#### Load libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import imdb
import Levenshtein
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import MaxAbsScaler

#### Load data

In [2]:
df_movie = pd.read_csv('data/movies.csv', index_col='imdbId')
df_description = pd.read_csv('data/movies_description.csv')

In [3]:
# Clean the data set
df_movie['year'] = df_movie['title'].str.extract(r'\((\d{4})\)')
df_movie['title'] = df_movie['title'].str.replace(r'\(\d{4}\)', '', regex=True).str.strip()
df_movie['year'] = pd.to_numeric(df_movie['year'], errors='coerce').fillna(0).astype(int)

In [4]:
print(df_movie)

         movieId                        title  \
imdbId                                          
114709         1                    Toy Story   
113497         2                      Jumanji   
113228         3             Grumpier Old Men   
114885         4            Waiting to Exhale   
113041         5  Father of the Bride Part II   
...          ...                          ...   
466713    131254        Kein Bund für's Leben   
277703    131256       Feuer, Eis & Dosenbier   
3485166   131258                  The Pirates   
249110    131260                 Rentun Ruusu   
1724965   131262                    Innocence   

                                              genres  year  
imdbId                                                      
114709   Adventure|Animation|Children|Comedy|Fantasy  1995  
113497                    Adventure|Children|Fantasy  1995  
113228                                Comedy|Romance  1995  
114885                          Comedy|Drama|Romance  199

In [5]:
print(df_movie.loc[88763])

movieId                       1270
title           Back to the Future
genres     Adventure|Comedy|Sci-Fi
year                          1985
Name: 88763, dtype: object


#### Feature engineering

In [ ]:
# Get dummy values
genre_features = df_movie['genres'].str.get_dummies(sep='|')

# Normalize year
scaler = MaxAbsScaler()
year_features = scaler.fit_transform(df_movie[['year']])

# Vectorize words
tfidf = TfidfVectorizer(stop_words='english', max_features=100)
tfidf_features = tfidf.fit_transform(df_movie['title']).toarray()

features = np.hstack([genre_features.values, year_features, tfidf_features])

In [7]:
model_knn = NearestNeighbors(n_neighbors=10, metric='cosine', algorithm='brute')
model_knn.fit(features)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",10
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'brute'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None


In [ ]:
# --- Recommendation System Logic ---

selected_movie_indices = []  # List to keep track of user selections

def show_recommendations():
    """Function to display recommendations based on past selections"""
    if not selected_movie_indices:
        return
    
    # Calculate the average vector of all selected movies (User Profile)
    # This represents the user's overall preference across Genres, Year, and Title keywords
    user_profile = features[selected_movie_indices].mean(axis=0).reshape(1, -1)
    
    # Find the nearest neighbors using KNN (k=10 for recommendations)
    # We fetch 20 neighbors to ensure we have 10 after filtering out already selected movies
    distances, indices = model_knn.kneighbors(user_profile, n_neighbors=20)
    
    print("\n[Recommendations for You (based on KNN, k=10)]")
    count = 0
    for idx in indices.flatten():
        # Requirement: Don't recommend movies already in the selection list
        if idx not in selected_movie_indices:
            movie = df_movie.iloc[idx]
            # Requirement: Show the title of the movie
            print(f"{movie['title']} | Genres: {movie['genres']}")
            count += 1
        if count >= 10: 
            break

# --- Main Interaction Loop ---

# Requirement: Recommend 10 random movies at the start
print("--- Initial Recommendations (Random 10) ---")
print(df_movie[['title', 'genres']].sample(10))

while True:
    # Requirement: Show what the user has selected
    current_titles = df_movie.loc[selected_movie_indices, 'title'].tolist()
    print(f"\nYour Current Selection: {current_titles}")
    
    # Requirement: Search feature based on movie title or year
    query = input("\nSearch by movie title or year (type 'q' to quit): ")
    if query.lower() == 'q': 
        break
    
    # Filtering the dataframe based on the search query
    results = df_movie[df_movie['title'].str.contains(query, case=False, na=False)]
    
    if results.empty:
        print("No movies found. Please try a different keyword.")
        continue
    
    # Requirement: Show search results (limited to top 10 for clarity)
    print(f"\n--- Search Results for '{query}' ---")
    print(results[['title', 'genres']].head(10))
    
    # Requirement: Let the user select movies
    choice = input("\nEnter the Index number to add to your list (or press Enter to skip): ")
    
    if choice.isdigit() and int(choice) in results.index:
        idx = int(choice)
        if idx not in selected_movie_indices:
            selected_movie_indices.append(idx)
            
            # Requirement: Show recommendations after each selection
            # Requirement: Use K-Nearest Neighbor algorithm to suggest movies
            show_recommendations()
        else:
            print("This movie is already in your list.")

--- Initial Recommendations (Random 10) ---
                                            title                   genres
imdbId                                                                    
50783    Nights of Cabiria (Notti di Cabiria, Le)                    Drama
271537                         Happy Here and Now                    Drama
110066                                 Houseguest                   Comedy
337824       And Starring Pancho Villa as Himself  Action|Comedy|Drama|War
127626                                      Kiler             Comedy|Crime
97236                        Dream a Little Dream     Comedy|Drama|Romance
72764                             Carry on Behind                   Comedy
1555093                      Seasoning House, The          Horror|Thriller
913425        Broken Embraces (Los abrazos rotos)   Drama|Romance|Thriller
93058                           Full Metal Jacket                Drama|War

Your Current Selection: []

--- Search Results for '' -

### Recommendation engine

In [122]:
all_genres = set([g for sublist in df_movie['genres'] for g in sublist])
genre_list = sorted(list(all_genres))

genre_data = []
for genres in df_movie['genres']:
    row = [1 if g in genres else 0 for g in genre_list]
    genre_data.append(row)

genre_features = np.array(genre_data)

In [123]:
tfidf_vectorizer = TfidfVectorizer()
word_vector = tfidf_vectorizer.fit_transform(df_movie['genres'])

In [124]:
# K nearest neighbor function
def metric_stub(base_case_value, comparator_value):
    return 0

def euclidean_distance(base_case_value, comparator_value):
    return abs(int(base_case_value) - int(comparator_value))

def jaccard_similarity(base_case_genres: str, compartor_genres: str):
    base_case_genres = set(base_case_genres.split('|'))
    compartor_genres = set(compartor_genres.split('|'))

    numerator = len(base_case_genres.intersection(compartor_genres))
    denomenator = len(base_case_genres.union(compartor_genres))
    return float(numerator) / float(denomenator)

def cosine_similarity_function(base_case_plot, comparator_plot):
    tfidf_matrix = tfidf_vectorizer.fit_transform((base_case_plot, comparator_plot))
    results = cosine_similarity(tfidf_matrix[0], tfidf_matrix[1])
    return results[0][0]

In [125]:
K = 10
base_case = df_movie.loc[114709]
print(f"Comparing all movies to our base case: {base_case['title']}.")

Comparing all movies to our base case: Toy Story.


In [126]:
comparison_type = 'genres'
# df_movie['metric'] = df_movie[comparison_type].map(lambda x: euclidean_distance(base_case[comparison_type], x))
# df_movie['jaccard'] = df_movie[comparison_type].map(lambda x: jaccard_similarity(base_case[comparison_type], x))
# df_movie['levenshtein'] = df_movie[comparison_type].map(lambda x: Levenshtein.distance(base_case[comparison_type], x))
df_movie['cosine'] = df_movie[comparison_type].map(lambda x: cosine_similarity_function(base_case[comparison_type], x))

In [127]:
df_movie.head()

,movieId,title,genres,year,cosine
imdbId,,,,,
114709,1,Toy Story,Adventure|Animation|Children|Comedy|Fantasy,1995,1.000000
113497,2,Jumanji,Adventure|Children|Fantasy,1995,0.656973
113228,3,Grumpier Old Men,Comedy|Romance,1995,0.194314
114885,4,Waiting to Exhale,Comedy|Drama|Romance,1995,0.150640
113041,5,Father of the Bride Part II,Comedy,1995,0.335176


In [128]:
sorted_df_movie = df_movie.sort_values(by='cosine')
sorted_df_movie.drop(88763, inplace=True)
print(sorted_df_movie['title'].head(K))

imdbId
83745      Come Back to the Five and Dime, Jimmy Dean, Ji...
2109059                                        Vietnam in HD
3028412                               Por un puñado de besos
1242522                           Oscar and the Lady in Pink
45507              Older Brother, Younger Sister (Ani imôto)
114746                    Twelve Monkeys (a.k.a. 12 Monkeys)
112792                                       Dangerous Minds
115012         Shanghai Triad (Yao a yao yao dao waipo qiao)
1038685                                King of Fighters, The
114117                                            Persuasion
Name: title, dtype: str


### Search engine

In [129]:
search_query = input("Enter the movie title or released year:")

results = df_movie[
    df_movie['title'].str.contains(search_query, case=False, na=False) |
    df_movie['year'].astype(str).str.contains(search_query, na=False)
]

if not results.empty:
    print(f"\nSearch results of {search_query}:")
    display_df = results[['title', 'year', 'genres']].head(30)
    print(display_df)
    if (len(results) > 10):
        print("The search results are truncated. It displays only 30 rows.")

else:
    print(f"Does not find movies")


Search results of 1:
                                                    title  year  \
imdbId                                                            
114709                                          Toy Story  1995   
113497                                            Jumanji  1995   
113228                                   Grumpier Old Men  1995   
114885                                  Waiting to Exhale  1995   
113041                        Father of the Bride Part II  1995   
113277                                               Heat  1995   
114319                                            Sabrina  1995   
112302                                       Tom and Huck  1995   
114576                                       Sudden Death  1995   
113189                                          GoldenEye  1995   
112346                            American President, The  1995   
112896                        Dracula: Dead and Loving It  1995   
112453                                  